# FahMai RAG — Improved Version

ปรับปรุง 8 จุดหลักจาก Starter Kit:
1. **Structured Chunking** — แบ่ง Chunk ตาม Markdown Header
2. **Metadata Enrichment** — เติมแบรนด์ + หมวดหมู่ในทุก Chunk
3. **Question Extraction** — สกัดคำถามจริงจาก noisy input
4. **Multi-Query Expansion** — LLM สร้างคำถามหลายรูปแบบ
5. **Query Router** — บังคับดึงไฟล์นโยบาย/FAQ ตาม intent
6. **Product Focus Filter** — ถ้าถามสินค้าเฉพาะตัว เน้นไฟล์นั้น ★ NEW
7. **Cross-Encoder Reranking** — จัดลำดับ Chunks ซ้ำ
8. **Sibling Chunk Retrieval** — ดึง chunk พี่น้องจากไฟล์เดียวกัน

**เพิ่มเติม:**
- System Prompt แบบ XML (English tags + Thai instructions)
- Pydantic output validation
- ใช้โมเดล `openthaigpt`

In [2]:
# Load data
!unzip super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

Archive:  super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
replace data/knowledge_base/policies/cancellation_policy.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: Archive:  super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
replace data/knowledge_base/policies/cancellation_policy.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [3]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests pydantic

In [4]:
# === CONFIGURATION ===
N_QUESTIONS   = 100        # เปลี่ยนเป็น 100 ตอน submit จริง
DATA_DIR      = "/content/data"
KB_DIR        = f"{DATA_DIR}/knowledge_base"
TOP_K         = 5         # chunks สุดท้ายที่ส่งให้ LLM
TOP_K_FETCH   = 15        # ดึงมาก่อน rerank
LLM_MODEL     = "openthaigpt"  # เปลี่ยนโมเดลได้: typhoon, openthaigpt, pathumma, kbtg

---
## Section 0: Setup & LLM + Pydantic

In [5]:
import os, csv, re, time, requests
import numpy as np
import re
from pathlib import Path
from typing import Optional
from pydantic import BaseModel, Field, field_validator
from google.colab import userdata

THAILLM_API_KEY = userdata.get('THAILLM_API_KEY')

In [6]:
def ask_llm(messages, model="openthaigpt", max_retries=5):
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {"model": "/model", "messages": messages, "temperature": 0}

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=60)
            if resp.status_code in [429, 502, 504]:
                time.sleep(2 ** attempt)
                continue
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            time.sleep(2 ** attempt)

    # ★ THE API RESCUE: ถ้าล่มครบ 5 รอบ ห้ามยอมแพ้ ให้สลับโมเดล!
    if model == "pathumma":
        print("    [API] Pathumma Dead! Switching to Typhoon as last resort...")
        return ask_llm(messages, model="typhoon", max_retries=2)
    elif model == "openthaigpt":
        print("    [API] OpenThaiGPT Dead! Switching to Typhoon...")
        return ask_llm(messages, model="typhoon", max_retries=2)

    return None

### Pydantic Output Validation

ใช้ Pydantic model validate output จาก LLM — ไม่ต้อง regex เอง ได้ทั้ง answer + reasoning สำหรับ debug

In [7]:
class RAGAnswer(BaseModel):
    """Validate & parse LLM output สำหรับ FahMai Q&A"""
    answer: int = Field(..., ge=1, le=10)
    reasoning: Optional[str] = None
    raw_response: str = ""

    @field_validator("answer")
    @classmethod
    def answer_in_range(cls, v):
        if not 1 <= v <= 10:
            raise ValueError(f"Answer {v} out of range 1-10")
        return v

    @classmethod
    def from_llm_response(cls, raw: Optional[str]) -> "RAGAnswer":
        if raw is None:
            return cls(answer=1, reasoning="LLM returned None", raw_response="")

        # ลบ <think>...</think>
        clean = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()

        # แยก reasoning
        reasoning = None
        r_match = re.search(r"<reasoning>(.+?)</reasoning>", clean, re.DOTALL)
        if r_match:
            reasoning = r_match.group(1).strip()
        else:
            # fallback: จับข้อความก่อน ANSWER:
            r_match2 = re.search(r"REASON(?:ING)?:\s*(.+?)(?=ANSWER:)", clean, re.DOTALL)
            if r_match2:
                reasoning = r_match2.group(1).strip()

        # Parse ANSWER: X
        a_match = re.search(r"ANSWER:\s*(\d+)", clean)
        if a_match:
            ans = int(a_match.group(1))
            if 1 <= ans <= 10:
                return cls(answer=ans, reasoning=reasoning, raw_response=raw)

        # Fallback: เลข 1-10 ตัวแรก
        for d in re.findall(r"\b(\d{1,2})\b", clean):
            if 1 <= int(d) <= 10:
                return cls(answer=int(d), reasoning=f"Fallback: found {d}", raw_response=raw)

        return cls(answer=1, reasoning="Could not parse", raw_response=raw)


def parse_answer(text):
    """Shortcut: parse → int"""
    return RAGAnswer.from_llm_response(text).answer


# === Test ===
test = RAGAnswer.from_llm_response("<reasoning>เจอข้อมูลว่าราคา 15,900</reasoning>\nANSWER: 5")
print(f"Test: answer={test.answer}, reasoning={test.reasoning}")

Test: answer=5, reasoning=เจอข้อมูลว่าราคา 15,900


### Test LLM (no RAG)

In [8]:
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ LLM ไม่รู้ข้อมูล FahMai → ต้องใช้ RAG!")

LLM response (no context): <think>
ผู้ใช้ถามเกี่ยวกับความสามารถในการกันน้ำของ S3 Ultra ซึ่งเป็นเรื่องสำคัญสำหรับผู้ใช้ที่ต้องการใช้งานในสภาพแวดล้อมที่เปียกชื้น
</think>

## ความสามารถในการกันน้ำของ S3 Ultra

**S3 Ultra** มีความสามารถในการกันน้ำตามมาตรฐาน IP68

### รายละเอียดความสามารถในการกันน้ำ

| รายการ | รายละเอียด |
|--------|------------|
| **มาตรฐาน** | IP68 |
| **ความลึก** | 1.5 เมตร |
| **ระยะเวลา** | 30 นาที |
| **กันน้ำได้** | ใช่ |
| **กันฝุ่นได้** | ใช่ (IP68) |

### หมายเหตุสำคัญ

1. **ไม่ได้รับการรับรอง** สำหรับการดำน้ำหรือใช้งานในน้ำลึก
2. **ไม่ควรใช้งาน** ในสภาพแวดล้อมที่มีความดันสูง เช่น ใต้น้ำลึก
3. **ควรปิดฝา** ทุกครั้งที่ไม่ได้ใช้งาน
4. **หลีกเลี่ยงการสัมผัส** กับน้ำยาทำความสะอาดหรือสารเคมี

### คำแนะนำ

- หลีกเลี่ยงการใช้งานในสภาพแวดล้อมที่เปียกชื้นมากเกินไป
- ทำความสะอาดและเช็ดให้แห้งทุกครั้งหลังใช้งานในน้ำ
- ตรวจสอบสภาพตัวเครื่องและช่องเสียบอย่างสม่ำเสมอ

**S3 Ultra** ออกแบบมาให้ทนทานต่อสภาพแวดล้อมทั่วไป แต่ไม่ได้ผลิตมาเพื่อใช้งานในน้ำอย่างต่อเนื่อง

→ LLM ไม่รู้

---
## Section 1: Load & Structured Chunking + Metadata

In [9]:
# === Load Documents ===
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3


In [10]:
# === Brand & Category Detection ===
# เพิ่มชื่อแบรนด์ตามที่เจอในข้อมูลจริง
KNOWN_BRANDS = [
    "SaiFah", "DaoNuea", "FahMai", "KluenSiang", "WongKhoJon", "Judchuam",
    "ArcWave", "NovaTech", "PulseGear", "ZenByte"
]

# ★ Thai ↔ English mapping
# สำคัญมาก: ลูกค้าพิมพ์ไทย แต่ filename เป็นอังกฤษ
# เปิด knowledge_base แล้วเพิ่มให้ครบทุกชื่อที่เจอ!
THAI_TO_ENG_MAP = {
    # === แบรนด์ ===
    "สายฟ้า": "saifah",
    "ดาวเหนือ": "daonuea",
    "ฟ้าใหม่": "fahmai",
    "จุดเชื่อม": "judchuam",
    "วงโคจร": "wongkhojon",
    "คลื่นเสียง": "kluensiang",
    "อาร์คเวฟ": "arcwave",
    "โนวาเทค": "novatech",
    "พัลส์เกียร์": "pulsegear",
    "เซนไบต์": "zenbyte",
    # === ชื่อรุ่นและประเภท ===
    "เฮดโปร": "headpro",
    "เฮดออน": "headon",
    "แอร์บุ๊ก": "airbook", "แอร์บุค": "airbook",
    "โปรบุ๊ก": "probook", "โปรบุค": "probook",
    "สลิมบุ๊ก": "slimbook", "สลิมบุค": "slimbook",
    "รักเก็ด": "rugged",
    "วอทช์": "watch",
    "แบนด์": "band"
}

ENG_TO_THAI_MAP = {v: k for k, v in THAI_TO_ENG_MAP.items()}


def normalize_query(text):
    """
    แปลงชื่อไทยเป็นอังกฤษ เพื่อให้ match กับ filename ได้
    ★ เรียงจากคำยาวไปสั้น เพื่อไม่ให้ 'แท็บ' แทน 'แท็บเล็ต' ก่อน
    """
    result = text.lower()
    # เรียงจากคำยาวก่อน
    sorted_items = sorted(THAI_TO_ENG_MAP.items(), key=lambda x: len(x[0]), reverse=True)
    for thai, eng in sorted_items:
        result = result.replace(thai.lower(), eng)
    return result


CATEGORY_KEYWORDS = {
    "สมาร์ทโฟน": ["phone", "โทรศัพท์", "มือถือ", "smartphone", "SP"],
    "แท็บเล็ต": ["tablet", "แท็บเล็ต", "tab", "TB"],
    "แล็ปท็อป": ["laptop", "โน้ตบุ๊ก", "notebook", "LT"],
    "เดสก์ท็อป": ["desktop", "คอมพิวเตอร์", "pc", "DT", "tower", "all in one"], # ★ เพิ่ม
    "สมาร์ทวอทช์": ["watch", "นาฬิกา", "smartwatch", "วอทช์", "SW", "FT", "band"],
    "หูฟัง/ลำโพง": ["earbuds", "หูฟัง", "buds", "headphone", "soundbar", "speaker", "EB", "HP", "SK"],
    "อุปกรณ์เสริม": ["charger", "cable", "case", "hub", "ชาร์จ", "สาย", "เคส", "CH", "CS", "HB", "mouse", "keyboard", "ram", "ssd"], # ★ เพิ่ม
    "นโยบาย": ["นโยบาย", "policy", "การรับประกัน", "คืนสินค้า", "เปลี่ยนสินค้า", "warranty"],
    "ข้อมูลร้าน": ["สาขา", "ร้าน", "store", "ที่อยู่", "เปิด-ปิด", "ติดต่อ", "faq", "guide"],
}


def detect_brands(text):
    text_lower = text.lower()
    found = [b for b in KNOWN_BRANDS if b.lower() in text_lower]
    for thai, eng in THAI_TO_ENG_MAP.items():
        if thai in text:
            for b in KNOWN_BRANDS:
                if eng.lower() in b.lower() and b not in found:
                    found.append(b)
    return found


def detect_category(text):
    text_lower = text.lower()
    for cat, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            if kw.lower() in text_lower:
                return cat
    return "general"


# === Test ===
print("Normalize tests:")
print(f"  'สายฟ้า โฟน X9 Pro Max' → '{normalize_query('สายฟ้า โฟน X9 Pro Max')}'")
print(f"  'คลื่นเสียง เฮดโปร X1' → '{normalize_query('คลื่นเสียง เฮดโปร X1')}'")
print(f"  'ดาวเหนือ แอร์บุ๊ก 14' → '{normalize_query('ดาวเหนือ แอร์บุ๊ก 14')}'")
print(f"\nCategory tests:")
print(f"  'หูฟังครอบหูตัดเสียง' → {detect_category('หูฟังครอบหูตัดเสียง')}")
print(f"  'แอร์บุ๊ก 14 แบตกี่ชม' → {detect_category('แอร์บุ๊ก 14 แบตกี่ชม')}")

Normalize tests:
  'สายฟ้า โฟน X9 Pro Max' → 'saifah โฟน x9 pro max'
  'คลื่นเสียง เฮดโปร X1' → 'kluensiang headpro x1'
  'ดาวเหนือ แอร์บุ๊ก 14' → 'daonuea airbook 14'

Category tests:
  'หูฟังครอบหูตัดเสียง' → หูฟัง/ลำโพง
  'แอร์บุ๊ก 14 แบตกี่ชม' → general


In [11]:
# === Structured Chunking ===
# แบ่งตาม Markdown header แทน fixed-size

def structured_chunk(doc_text):
    """แบ่งตาม ## header, split ย่อยถ้ายาวเกิน 1500 chars"""
    MAX_CHUNK = 1500
    OVERLAP   = 200

    sections = re.split(r'(?=^#{1,3}\s)', doc_text, flags=re.MULTILINE)
    sections = [s.strip() for s in sections if s.strip()]

    if len(sections) <= 1:
        sections = [doc_text.strip()]

    result = []
    for section in sections:
        if len(section) <= MAX_CHUNK:
            result.append(section)
        else:
            start = 0
            while start < len(section):
                result.append(section[start:start + MAX_CHUNK])
                start += MAX_CHUNK - OVERLAP
    return result


# === Build Chunks with Metadata ===
chunks = []
for doc in documents:
    raw_chunks = structured_chunk(doc["text"])
    doc_brands = detect_brands(doc["text"])
    category = detect_category(doc["text"])
    first_line = doc["text"].strip().split("\n")[0]
    doc_title = re.sub(r'^#+\s*', '', first_line).strip()

    for chunk_text in raw_chunks:
        all_brands = list(set(doc_brands + detect_brands(chunk_text)))

        # Metadata prefix สำหรับ retrieval
        meta = ""
        if all_brands:
            meta += f"[แบรนด์: {', '.join(all_brands)}] "
        if category != "general":
            meta += f"[หมวด: {category}] "
        if doc_title:
            meta += f"[เอกสาร: {doc_title}] "

        chunks.append({
            "text": meta + chunk_text,    # enriched (สำหรับ retrieval)
            "raw_text": chunk_text,        # original (สำหรับส่งให้ LLM)
            "source": doc["path"],
            "brands": all_brands,
            "category": category,
            "doc_title": doc_title,
        })

print(f"Created {len(chunks)} structured chunks from {len(documents)} documents")
print(f"\n--- Sample ---")
print(f"Source: {chunks[0]['source']}")
print(f"Brands: {chunks[0]['brands']}, Category: {chunks[0]['category']}")
print(chunks[0]['text'][:400])

Created 890 structured chunks from 118 documents

--- Sample ---
Source: policies/cancellation_policy.md
Brands: ['FahMai'], Category: สมาร์ทโฟน
[แบรนด์: FahMai] [หมวด: สมาร์ทโฟน] [เอกสาร: นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่] # นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---


---
## Section 2: Embedding + BM25 Index

In [12]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
print(f"Embeddings shape: {chunk_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embeddings shape: (890, 384)


In [13]:
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 index: {len(tokenized_chunks)} chunks")

BM25 index: 890 chunks


---
## Section 3: Retrieval (Dense + BM25 + Hybrid)

In [14]:
def dense_retrieve(query, chunk_embs, k=TOP_K_FETCH):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]


def bm25_retrieve(query, k=TOP_K_FETCH):
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]


def hybrid_retrieve(query, chunk_embs, k=TOP_K_FETCH, rrf_k=60):
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

---
## Section 4: Question Extraction + Multi-Query Expansion

**แก้ Root Cause #1**: คำถามยาวมี noise (เรื่องเที่ยว/ร้านอาหาร) → สกัดคำถามจริงก่อน expand

Flow: `Noisy Question → Extract Core → Expand 3 variants → Hybrid Retrieve`

In [15]:
# === Step A: Question Extraction ===
# สกัดคำถามจริงจากข้อความยาวที่มี noise

EXTRACT_PROMPT = """<task>
คุณเป็นผู้ช่วยสกัดคำถามหลัก จากข้อความของลูกค้าที่อาจมีเรื่องเล่า/บริบทยาวๆ ปนมา
</task>

<rules>
- สกัดเฉพาะ "คำถามจริง" ที่ลูกค้าต้องการคำตอบ
- ตัดเรื่องเล่า, สถานที่ท่องเที่ยว, ร้านอาหาร, เพื่อน, แพลนเดินทาง ออกให้หมด
- คงชื่อสินค้า, แบรนด์, รุ่น, สเปค ไว้ครบทุกตัว
- ถ้ามีหลายคำถามย่อย ให้รวมเป็นประโยคกระชับ คั่นด้วย " และ "
- ตอบเฉพาะคำถามที่สกัดแล้ว บรรทัดเดียว ไม่ต้องอธิบาย
</rules>

<example>
ข้อความ: เฮ้ย จะไปเที่ยวภูเก็ต จองโรงแรมไว้แล้ว มีจุดดำน้ำสวยมาก อยากรู้ว่า SaiFah Rugged R1 กันน้ำระดับไหน เอาไปดำน้ำได้มั้ย แล้วถ้าน้ำเค็มเข้าประกันคุ้มครองมั้ย
คำถามหลัก: SaiFah Rugged R1 กันน้ำระดับไหน ดำน้ำถ่ายรูปใต้ทะเลได้หรือไม่ และถ้าน้ำเค็มเข้าเครื่องประกันคุ้มครองหรือไม่
</example>

ข้อความ: {question}
คำถามหลัก:"""


def extract_core_question(question):
    """สกัดคำถามจริงจากข้อความยาว (ข้อความสั้น <100 chars → ใช้เลย)"""
    if len(question) < 100:
        return question

    raw = ask_llm([{"role": "user", "content": EXTRACT_PROMPT.format(question=question)}])

    if raw:
        # ★ ลบ <think>...</think> ก่อน
        cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
        # ★ ลบ tag เปิดที่ไม่มี tag ปิด (กรณี LLM ส่ง <think> แต่ไม่ปิด)
        cleaned = re.sub(r'<think>.*', '', cleaned, flags=re.DOTALL).strip()

        if len(cleaned) > 10:
            extracted = cleaned.split("\n")[0].strip()
            extracted = re.sub(r'^คำถามหลัก:\s*', '', extracted).strip()
            # ★ ตรวจว่า extracted ไม่ใช่ garbage
            if len(extracted) > 10 and not extracted.startswith('<'):
                print(f"    Extracted: {extracted[:80]}")
                return extracted

    print(f"    Extract failed, using original")
    return question

In [16]:
# === Step B: Expand Query ===

EXPAND_PROMPT = """<task>
คุณเป็นผู้เชี่ยวชาญค้นหาข้อมูลร้าน FahMai Electronics (ขายเฉพาะแบรนด์ SaiFah, DaoNuea, KluenSiang, WongKhoJon, JudChuem, ZenByte, NovaTech, PulseGear, ArcWave เท่านั้น)
หน้าที่ของคุณคือแปลงคำถามของลูกค้า ให้เป็น "คำค้นหา (Search Query)" 3 รูปแบบ
</task>

<rules>
1. คงชื่อรุ่นสินค้า แบรนด์ และรหัสสินค้าไว้ให้ครบถ้วนเสมอ
2. แปลงภาษาพูดเป็นคีย์เวิร์ดทางการที่มักปรากฏในคู่มือหรือสเปค
3. ★ หากลูกค้าถามถึงฟังก์ชัน/ความต้องการ ให้เติม "คำศัพท์ทางเทคนิค" ที่เกี่ยวข้องลงไปด้วย
4. ★ ห้ามเดาชื่อแบรนด์ภายนอก (เช่น iPhone, Apple, Samsung) เด็ดขาด! หากลูกค้าพิมพ์แค่ชื่อย่อหรือชื่อรุ่น (เช่น "แบนด์ 8", "X9") ให้ใช้คำนั้นตรงๆ หรือจับคู่กับแบรนด์ในร้าน (เช่น WongKhoJon Band 8, SaiFah X9)
5. หากคำถามเป็นการตามหาสินค้า สเปค หรือฟังก์ชัน ให้เพิ่มคำว่า "สเปคทางเทคนิค" และ "รายละเอียดสินค้า" ต่อท้าย
</rules>

ตอบเฉพาะ 3 คำค้นหา คั่นด้วยบรรทัดใหม่ ไม่ต้องใส่หมายเลข ไม่ต้องอธิบาย

คำถามต้นฉบับ: {question}"""


def expand_query(question):
    """Extract core question → expand เป็นหลาย queries"""

    # สมมติว่าคุณมีฟังก์ชันนี้อยู่แล้ว ถ้าไม่มีให้ใช้ clean_question = question ไปก่อน
    clean_question = question
    try:
        if 'extract_core_question' in globals():
            clean_question = extract_core_question(question)
    except Exception:
        pass

    raw = ask_llm([{"role": "user", "content": EXPAND_PROMPT.format(question=clean_question)}])

    queries = [clean_question]

    if clean_question != question:
        queries.append(question)

    if raw:
        # ลบ <think>...</think> ของ Pathumma
        cleaned = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()

        for line in cleaned.split("\n"):
            line = re.sub(r'^\d+[\.)\-]\s*', '', line).strip()

            if not line or len(line) <= 5:
                continue
            if line in queries:
                continue
            # กรองคำอธิบายขยะจาก LLM
            if any(bad in line.lower() for bad in [
                'กฎ', 'กฏ', 'ปฏิบัติ', 'rules', '<think', '</think',
                'คำถามต้นฉบับ', 'หมายเหตุ', 'อธิบาย', 'ตัวอย่าง',
                'แปลงภาษา', 'synonym', 'คำค้น'
            ]):
                continue

            # ★ ปลดล็อก Regex คลั่งคำถามออก! ให้รับ Query ที่เป็นประโยคบอกเล่าหรือกลุ่มคำได้
            queries.append(line)

    # คืนค่า original + expanded (จำกัดแค่ 4-5 อันเพื่อไม่ให้ API หน่วงเกินไป)
    return queries[:5]


def multi_query_hybrid_retrieve(question, chunk_embs, k=TOP_K_FETCH):
    """Multi-Query (with extraction) + Hybrid → RRF fusion"""
    queries = expand_query(question)
    print(f"    Expanded: {len(queries)} queries")
    for i, q in enumerate(queries):
        print(f"      q{i+1}: {q[:80]}{'...' if len(q) > 80 else ''}")

    combined_rrf = {}
    rrf_k = 60

    for q in queries:
        d_idx, _ = dense_retrieve(q, chunk_embs, k=k)
        for rank, idx in enumerate(d_idx, 1):
            combined_rrf[idx] = combined_rrf.get(idx, 0) + 1.0 / (rrf_k + rank)

        b_idx, _ = bm25_retrieve(q, k=k)
        for rank, idx in enumerate(b_idx, 1):
            combined_rrf[idx] = combined_rrf.get(idx, 0) + 1.0 / (rrf_k + rank)

    sorted_idx = sorted(combined_rrf.keys(), key=lambda x: combined_rrf[x], reverse=True)[:k]
    return sorted_idx


# Demo
print("=== Short question (no extraction needed) ===")
print(expand_query("Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"))

=== Short question (no extraction needed) ===
['Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?', 'Watch S3 Ultra สเปคทางเทคนิค ความลึกการกันน้ำ', 'Watch S3 Ultra รายละเอียดสินค้า ค่าความดันน้ำ ATM', 'Watch S3 Ultra ความสามารถกันน้ำ สเปคทางเทคนิค รายละเอียดสินค้า']


---
## Section 5: Cross-Encoder Reranking + Sibling Chunks

**แก้ Root Cause #2**: Structured chunking แยก `## สเปค` กับ `## ประกัน` คนละ chunk

→ หลัง rerank เลือก top-k แล้ว ดึง chunk พี่น้องจากไฟล์เดียวกันมาด้วย

In [17]:
from sentence_transformers import CrossEncoder

# ★ THE FIX: เปลี่ยนมาใช้โมเดล Multilingual ที่เก่งภาษาไทย (แนะนำ m3)
# ถ้า bge-reranker-v2-m3 ใหญ่ไป ให้ใช้ BAAI/bge-reranker-base ตามที่คุณคอมเมนต์ไว้
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def retrieve_sibling_chunks(selected_indices, all_chunks, window=1):
    """
    ดึง Chunk บริบทที่อยู่ติดกัน (บน-ล่าง) จากไฟล์เดียวกัน
    ทำงานเร็วและกินทรัพยากรน้อยลงด้วยการเช็ค Index ตรงๆ (index - 1, index + 1)
    """
    combined_set = set(selected_indices)

    for idx in selected_indices:
        src = all_chunks[idx]["source"]

        # 1. เช็ค Chunk ก่อนหน้า (ด้านบน)
        for i in range(1, window + 1):
            prev_idx = idx - i
            if prev_idx >= 0 and all_chunks[prev_idx]["source"] == src:
                combined_set.add(prev_idx)

        # 2. เช็ค Chunk ถัดไป (ด้านล่าง)
        for i in range(1, window + 1):
            next_idx = idx + i
            if next_idx < len(all_chunks) and all_chunks[next_idx]["source"] == src:
                combined_set.add(next_idx)

    # รวมและเรียงลำดับ Index เพื่อให้ LLM อ่านเนื้อหาได้ปะติดปะต่อกันเหมือนต้นฉบับ
    combined_list = sorted(list(combined_set), key=lambda x: (all_chunks[x]["source"], x))
    return combined_list


def rerank_chunks(question, candidate_indices, all_chunks, top_k=5):
    """Rerank → คัด Top K → ดึง Sibling chunks ประกบหัวท้าย"""

    if len(candidate_indices) <= top_k:
        reranked = list(candidate_indices)
    else:
        # ใช้ all_chunks ที่ส่งเข้ามาเป็นพารามิเตอร์ เพื่อป้องกันปัญหา Global Variable
        pairs = [(question, all_chunks[i]["text"]) for i in candidate_indices]
        scores = reranker.predict(pairs)

        scored = sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)
        reranked = [idx for idx, _ in scored[:top_k]]

    # ★ ดึง sibling chunks เพิ่ม (window=1 คือดึงบน 1 ก้อน ล่าง 1 ก้อน)
    with_siblings = retrieve_sibling_chunks(reranked, all_chunks, window=1)

    print(f"    Reranked {len(reranked)} → with siblings: {len(with_siblings)} chunks")
    return with_siblings

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

---
## Section 5.5: Query Router — บังคับดึงไฟล์ตาม intent

**ปัญหา**: Retrieval โฟกัสแต่ไฟล์สินค้า แต่คำตอบจริงอยู่ใน `warranty_policy.md` หรือ `general_faq.md`

**วิธีแก้**: ตรวจ intent ของคำถาม → บังคับดึง chunks จากไฟล์ที่เกี่ยวข้องเข้ามาเสมอ

| Intent | ไฟล์ที่ต้องดึง |
|--------|---------------|
| ทั่วไป/ไม่ระบุเจาะจง | `about_fahmai`, `general_faq` |
| ประกัน/ซ่อม/Care+ | `warranty_policy`, `return_policy` |
| ซื้อ/เปรียบเทียบ/ไม่รู้จะซื้ออะไร | `buying_guide_smartphones_2569` |
| จัดส่ง/ติดตามพัสดุ | `shipping_policy` |
| สมาชิก/Points | `membership_points_policy` |
| ยกเลิก/คืนเงิน | `cancellation_policy`, `return_policy` |

In [18]:
# === Query Router ===
# ตรวจ intent → บังคับดึง chunks จากไฟล์ที่เกี่ยวข้อง

INTENT_ROUTES = {
    "warranty": {
        "keywords": ["ประกัน", "รับประกัน", "warranty", "ซ่อม", "เคลม", "care+",
                     "care plus", "จอแตก", "เสีย", "พัง", "น้ำเข้า", "ตกน้ำ",
                     "กันน้ำ", "ชำรุด", "บกพร่อง"],
        "must_include": ["warranty_policy", "return_policy"],
    },
    "buying": {
        "keywords": ["ซื้ออะไรดี", "แนะนำ", "เปรียบเทียบ", "เทียบ", "ต่างกันยังไง",
                     "รุ่นไหนดี", "คุ้มไหม", "buying guide", "งบ", "ราคา",
                     "ไม่รู้จะซื้อ", "ตัวไหนดี", "เลือกยังไง"],
        "must_include": ["buying_guide"],
    },
    "shipping": {
        "keywords": ["จัดส่ง", "ส่งของ", "shipping", "ขนส่ง", "พัสดุ",
                     "tracking", "ติดตาม", "กี่วัน", "ค่าส่ง", "ส่งฟรี"],
        "must_include": ["shipping_policy"],
    },
    "membership": {
        "keywords": ["สมาชิก", "member", "points", "แต้ม", "คะแนน",
                     "สะสม", "ระดับ", "tier", "fahmai points"],
        "must_include": ["membership_points_policy"],
    },
    "cancel_return": {
        "keywords": ["ยกเลิก", "cancel", "คืนสินค้า", "คืนเงิน", "refund",
                     "เปลี่ยน", "ไม่พอใจ", "ส่งคืน", "return"],
        "must_include": ["cancellation_policy", "return_policy"],
    },
    "general": {
        "keywords": [],  # fallback — ใช้เสมอถ้าไม่ match intent อื่น
        "must_include": ["about_fahmai", "general_faq"],
    },
}


def detect_intent(question):
    """ตรวจ intent จาก keywords → return list ของ intent ที่ match"""
    question_lower = question.lower()
    matched = []

    for intent, config in INTENT_ROUTES.items():
        if intent == "general":
            continue  # general เป็น fallback
        for kw in config["keywords"]:
            if kw in question_lower:
                matched.append(intent)
                break

    # ถ้าไม่ match intent ไหนเลย → ใช้ general
    if not matched:
        matched.append("general")

    return matched


def get_mandatory_file_patterns(question):
    """จาก intent → return list ของ filename patterns ที่ต้องดึง"""
    intents = detect_intent(question)
    patterns = []
    for intent in intents:
        patterns.extend(INTENT_ROUTES[intent]["must_include"])
    return list(set(patterns))  # deduplicate


def inject_mandatory_chunks(question, existing_indices, all_chunks, max_per_file=2):
    """
    ดึง chunks จากไฟล์ mandatory ที่ยังไม่อยู่ใน existing_indices
    """
    patterns = get_mandatory_file_patterns(question)
    if not patterns:
        return existing_indices

    intents = detect_intent(question)
    print(f"    Intent: {intents} → must include: {patterns}")

    # ★ THE FIX 2: ถ้าเป็นคำถาม general ปกติ ให้ดึง FAQ แค่ไฟล์ละ 1 ก้อนพอ (ป้องกันไปเบียดไฟล์สินค้า)
    # แต่ถ้าเป็นคำถามประกัน/คืนของ ให้ดึงไฟล์ละ 2-3 ก้อนตามปกติ
    current_max = 1 if intents == ["general"] else max_per_file

    existing_set = set(existing_indices)
    extra_indices = []

    # ★ THE FIX 1: ย้ายการคำนวณ Embedding ออกมานอกลูป ทำแค่ครั้งเดียว!
    q_emb = embed_model.encode([question], normalize_embeddings=True)

    for pattern in patterns:
        # หา chunks จากไฟล์ที่ match pattern
        matching_chunks = [
            i for i, c in enumerate(all_chunks)
            if pattern.lower() in c["source"].lower() and i not in existing_set
        ]

        if not matching_chunks:
            continue

        # เลือก chunk ที่ relevant ที่สุดจากไฟล์นี้ (by dense similarity)
        # ใช้ matching_chunks เพื่อดึง index ไปเทียบใน chunk_embeddings (ซึ่งเป็น Global Variable)
        match_embs = chunk_embeddings[matching_chunks]
        scores = np.dot(match_embs, q_emb.T).flatten()

        # ดึงตามโควต้า current_max
        top_in_file = np.argsort(scores)[::-1][:current_max]
        for ti in top_in_file:
            extra_indices.append(matching_chunks[ti])

    if extra_indices:
        print(f"    Injected {len(extra_indices)} mandatory chunks from: {patterns}")

    # รวม existing + mandatory, deduplicate แต่ยังรักษาลำดับ
    combined = list(dict.fromkeys(list(existing_indices) + extra_indices))
    return combined


# === Test ===
print("Q: 'จอมือถือแตก ซ่อมได้มั้ย ถ้าซื้อ Care+ ไว้'")
print(f"  Intent: {detect_intent('จอมือถือแตก ซ่อมได้มั้ย ถ้าซื้อ Care+ ไว้')}")
print(f"  Must include: {get_mandatory_file_patterns('จอมือถือแตก ซ่อมได้มั้ย ถ้าซื้อ Care+ ไว้')}")
print()
print("Q: 'มือถือราคาไม่เกิน 15000 ตัวไหนดี'")
print(f"  Intent: {detect_intent('มือถือราคาไม่เกิน 15000 ตัวไหนดี')}")
print(f"  Must include: {get_mandatory_file_patterns('มือถือราคาไม่เกิน 15000 ตัวไหนดี')}")
print()
print("Q: 'ร้าน FahMai เปิดกี่โมง'")
print(f"  Intent: {detect_intent('ร้าน FahMai เปิดกี่โมง')}")
print(f"  Must include: {get_mandatory_file_patterns('ร้าน FahMai เปิดกี่โมง')}")

Q: 'จอมือถือแตก ซ่อมได้มั้ย ถ้าซื้อ Care+ ไว้'
  Intent: ['warranty']
  Must include: ['return_policy', 'warranty_policy']

Q: 'มือถือราคาไม่เกิน 15000 ตัวไหนดี'
  Intent: ['buying']
  Must include: ['buying_guide']

Q: 'ร้าน FahMai เปิดกี่โมง'
  Intent: ['general']
  Must include: ['about_fahmai', 'general_faq']


---
## Section 5.6: Product Focus Filter

**ปัญหา**: ถาม X9 Pro Max แต่ reranker ดันเลือก case Judchuam + รุ่นอื่นเข้ามาเต็ม

**สาเหตุ**: Cross-encoder ให้คะแนน `judchuam_case_clear_x9_pro` สูง เพราะมีคำว่า "X9 Pro" อยู่ด้วย

**วิธีแก้**: ตรวจว่าคำถามเจาะจงสินค้าตัวเดียวหรือไม่ ถ้าใช่ → จัดสรร slots ให้สินค้าหลักก่อน

| ตัวอย่างคำถาม | Focus file | ผลลัพธ์ |
|---|---|---|
| "สายฟ้า โฟน X9 Pro Max ผลิตที่ไหน" | `saifah_phone_x9_pro_max.md` | 4-5 slots จากไฟล์นี้ |
| "เปรียบเทียบ X9 กับ X9 Pro" | ไม่ focus (หลายสินค้า) | ปกติ |
| "ร้าน FahMai เปิดกี่โมง" | ไม่ focus (ไม่ใช่สินค้า) | ปกติ |

**ใช้ `THAI_TO_ENG_MAP`**: แปลง "สายฟ้า โฟน" → "saifah phone" ก่อน match กับ filename

In [19]:
# === Product Focus Filter ===
# สร้าง product catalog จาก chunks (ไม่ต้อง hardcode)

def build_product_catalog(all_chunks):
    """
    สร้าง catalog: {source_file: {product_name, brands, category, chunk_indices}}
    เอาไว้ match กับคำถามและดึง chunk ที่เหลือจากไฟล์เดียวกัน
    """
    catalog = {}
    for i, c in enumerate(all_chunks):
        src = c["source"]
        if src not in catalog:
            catalog[src] = {
                # ★ THE FIX: แก้ key ให้ตรงกับที่ find_focused_products ต้องการ
                "product_name": c.get("product_name", c.get("doc_title", "")),
                "brands": c.get("brands", []),
                "category": c.get("category", ""),
                "chunk_indices": [],
            }
        catalog[src]["chunk_indices"].append(i)
    return catalog


product_catalog = build_product_catalog(chunks)
print(f"Product catalog: {len(product_catalog)} files")
for src, info in list(product_catalog.items())[:5]:
    # อัปเดต print statement ให้ดึง product_name
    print(f"  {src}: {info['product_name'][:40]} ({len(info['chunk_indices'])} chunks)")

Product catalog: 118 files
  policies/cancellation_policy.md: นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ (17 chunks)
  policies/membership_points_policy.md: นโยบายสมาชิกและ FahMai Points — ร้านฟ้าใ (27 chunks)
  policies/return_policy.md: นโยบายการคืนสินค้าและการขอเงินคืน — ร้าน (23 chunks)
  policies/shipping_policy.md: นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ (17 chunks)
  policies/warranty_policy.md: นโยบายการรับประกันสินค้า — ร้านฟ้าใหม่ (26 chunks)


In [20]:
def find_focused_products(queries, catalog):
    if isinstance(queries, str):
        q_combined = queries.lower()
    else:
        q_combined = " ".join(queries).lower()

    product_scores = {}
    modifiers = ['max', 'ultra', 'plus', 'pro', 'mini', 'lite', 'se', 'wifi', 'edition', 'oled', 'bundle', 'fe']

    # ★ 1. Mapping คำไทยให้เป็นรหัส/ตระกูลสินค้า (แก้ Test 2)
    alias_map = {
        "ครอบหู": ["headpro", "headon", "hp"],
        "ตัดเสียง": ["pro", "anc"],
        "ซิม": ["phone", "sp"],
        "แบตสำรอง": ["powerbank", "pg", "pb"],
        "หัวชาร์จ": ["charger", "ch"],
        # ★ THE FIX: เติมคำศัพท์ตระกูล WongKhoJon
        "แบนด์": ["band", "wongkhojon", "ft"],
        "สายรัดข้อมือ": ["band", "wongkhojon", "ft"],
        "นาฬิกา": ["watch", "wongkhojon", "sw"],
        "วอทช์": ["watch", "wongkhojon", "sw"],
    }

    eng_words = re.findall(r'[a-z0-9]+', q_combined)

    # เติมคำภาษาอังกฤษเข้าไปถ้าระบบเจอคำไทยที่ตรงกับ Alias
    for thai_kw, eng_aliases in alias_map.items():
        if thai_kw in q_combined:
            eng_words.extend(eng_aliases)

    # ทำให้ลิสต์คำศัพท์ไม่ซ้ำกัน
    eng_words = list(set(eng_words))

    for src, info in catalog.items():
        if not src.startswith('products/'):
            continue

        filename = Path(src).stem.lower()
        clean_filename = filename.replace('-', ' ').replace('_', ' ')
        score = 0

        # 2. ให้คะแนนแบบ Exact Word Match (แม่นกว่าแบบ Substring)
        file_words = clean_filename.split()
        for word in eng_words:
            if len(word) >= 2:
                if word in file_words:
                    score += 2  # ชนเป๊ะๆ เป็นคำๆ ให้คะแนนเยอะ
                elif word in clean_filename:
                    score += 1  # เป็นแค่ส่วนประกอบของคำ ให้คะแนนน้อยลง

        # 3. Exact Product Name Match
        prod_name = info.get('product_name', '')
        if prod_name and prod_name.lower() in q_combined:
            score += 5

        # 4. Anti-Modifier Penalty
        for mod in modifiers:
            if f" {mod}" in clean_filename or clean_filename.endswith(mod):
                if mod not in q_combined and mod not in eng_words:
                    score -= 10
            elif mod in q_combined:
                if f" {mod}" not in clean_filename and not clean_filename.endswith(mod):
                    score -= 5

        # 5. กฎลงโทษอุปกรณ์เสริม (แก้ Test 4)
        phone_keywords = ["ซิม", "sim", "โทร", "call", "กล้อง", "camera", "จอ", "screen"]
        accessory_types = ["case", "film", "glass", "cable", "charger", "adapter", "hub"]

        is_phone_question = any(pk in q_combined for pk in phone_keywords)
        is_accessory_file = any(at in clean_filename for at in accessory_types)

        if is_phone_question and is_accessory_file:
            score -= 15

        is_acc_question = any(at in q_combined for at in accessory_types) or "เคส" in q_combined or "ฟิล์ม" in q_combined
        if is_accessory_file and not is_acc_question:
            score -= 5

        # 6. ★ THE KILL-SWITCH: ห้ามข้ามหมวดหมู่ (แก้ข้อ 17 โน้ตบุ๊ค vs SSD) ★
        laptop_kws = ["โน้ตบุ๊ค", "แล็ปท็อป", "notebook", "laptop"]
        is_laptop_q = any(kw in q_combined for kw in laptop_kws)

        # ถัาถามหาโน้ตบุ๊ค แต่ไฟล์ไม่ใช่ตระกูลโน้ตบุ๊ค (LT หรือมีคำว่า book) ให้หักคะแนนหนักๆ
        if is_laptop_q and "lt" not in clean_filename and "book" not in clean_filename:
            score -= 100

        if score > 0:
            product_scores[src] = score

    if not product_scores:
        return []

    sorted_sources = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)
    best_score = sorted_sources[0][1]

    top_products = []
    for src, score in sorted_sources:
        if score >= best_score - 4:
            top_products.append({"file": src, "score": score})
        if len(top_products) >= 3:
            break

    return top_products


def apply_product_focus(question, candidate_indices, all_chunks, top_k=TOP_K):
    """
    ★ รองรับทั้ง single product และ comparison (multi-product)
    """
    focused = find_focused_products(question, product_catalog)

    if not focused:
        print(f"    Product focus: ไม่เจาะจงสินค้า → ใช้ retrieval ปกติ")
        return candidate_indices

    focused_files = [f["file"] for f in focused]
    print(f"    Product focus: {focused_files}")
    for f in focused:
        print(f"      {f['file']} (score={f['score']})")

    # แยก candidates เป็น 3 กลุ่ม
    primary = []     # จากไฟล์สินค้าที่ focus
    policy = []      # จาก policy/FAQ/store_info
    other = []       # จากสินค้าอื่น

    for idx in candidate_indices:
        src = all_chunks[idx]["source"]
        if src in focused_files:
            primary.append(idx)
        elif src.startswith("policies/") or src.startswith("store_info/"):
            policy.append(idx)
        else:
            other.append(idx)

    # ★ ถ้า primary ขาด → ดึงจาก catalog สำหรับทุก focused file
    for f_info in focused:
        f_file = f_info["file"]
        has_chunks_from_file = any(
            all_chunks[idx]["source"] == f_file for idx in primary
        )
        if not has_chunks_from_file and f_file in product_catalog:
            all_from_file = product_catalog[f_file]["chunk_indices"]
            q_emb = embed_model.encode([question], normalize_embeddings=True)
            file_embs = chunk_embeddings[all_from_file]
            scores = np.dot(file_embs, q_emb.T).flatten()
            # จำกัด chunks ต่อไฟล์ตามจำนวน focused files
            per_file = max(3, top_k // len(focused))
            top_in_file = np.argsort(scores)[::-1][:per_file]
            new_chunks = [all_from_file[ti] for ti in top_in_file]
            primary.extend(new_chunks)
            print(f"    Forced retrieval from {f_file}: {len(new_chunks)} chunks")

    result = primary[:]

    intents = detect_intent(question)
    if any(i != "general" for i in intents):
        result.extend(policy[:3])
    else:
        result.extend(policy[:1])

    if len(result) < top_k:
        result.extend(other[:top_k - len(result)])

    print(f"    Focus result: {len(primary)} primary + {len(result)-len(primary)} others")
    return result


# === Test ===
print("Test 1: สายฟ้า โฟน X9 Pro Max ผลิตที่ประเทศอะไร (single focus)")
q1 = "สายฟ้า โฟน X9 Pro Max ผลิตที่ประเทศอะไรครับ"
print(f"  Normalized: '{normalize_query(q1)}'")
print(f"  → Focus: {find_focused_products(q1, product_catalog)}")
print()

print("Test 2: ★ Q13 หูฟังครอบหูตัดเสียง (ต้องเจอ kluensiang headpro)")
q2 = "ต้องการหูฟังครอบหูที่ตัดเสียงรบกวนได้ดี แนะนำรุ่นไหน"
print(f"  Normalized: '{normalize_query(q2)}'")
print(f"  Category: {detect_category(q2)}")
print(f"  → Focus: {find_focused_products(q2, product_catalog)}")
print()

print("Test 3: ★ Q24 เปรียบเทียบ 2 ตัว (ต้องได้ 2 files)")
q3 = "เปรียบเทียบแบตแล็ปท็อป ดาวเหนือ แอร์บุ๊ก 14 กับ NovaTech SlimBook 14"
print(f"  Normalized: '{normalize_query(q3)}'")
print(f"  → Focus: {find_focused_products(q3, product_catalog)}")
print()

print("Test 4: ซิม 2 ค่าย X9 Pro (ต้องเป็น phone ไม่ใช่ film!)")
q4 = "อยากใช้ซิม 2 ค่ายพร้อมกัน ใส่ซิมได้ 2 ใบมั้ย มองอยู่ X9 Pro"
print(f"  → Focus: {find_focused_products(q4, product_catalog)}")
print()

print("Test 5: ร้าน FahMai เปิดกี่โมง (ไม่ใช่สินค้า → ไม่ focus)")
print(f"  → Focus: {find_focused_products('ร้าน FahMai เปิดกี่โมง', product_catalog)}")

Test 1: สายฟ้า โฟน X9 Pro Max ผลิตที่ประเทศอะไร (single focus)
  Normalized: 'saifah โฟน x9 pro max ผลิตที่ประเทศอะไรครับ'
  → Focus: [{'file': 'products/SF-SP-001_saifah_phone_x9_pro_max.md', 'score': 6}, {'file': 'products/DN-LT-006_daonuea_probook_14_max.md', 'score': 3}]

Test 2: ★ Q13 หูฟังครอบหูตัดเสียง (ต้องเจอ kluensiang headpro)
  Normalized: 'ต้องการหูฟังครอบหูที่ตัดเสียงรบกวนได้ดี แนะนำรุ่นไหน'
  Category: หูฟัง/ลำโพง
  → Focus: [{'file': 'products/KS-HP-001_kluensiang_headpro_x1.md', 'score': 5}, {'file': 'products/KS-HP-003_kluensiang_headon_700.md', 'score': 4}, {'file': 'products/KS-HP-004_kluensiang_headon_500.md', 'score': 4}]

Test 3: ★ Q24 เปรียบเทียบ 2 ตัว (ต้องได้ 2 files)
  Normalized: 'เปรียบเทียบแบตแล็ปท็อป daonuea airbook 14 กับ novatech slimbook 14'
  → Focus: [{'file': 'products/NT-LT-001_novatech_slimbook_14.md', 'score': 6}, {'file': 'products/DN-LT-002_daonuea_airbook_14.md', 'score': 2}, {'file': 'products/DN-LT-003_daonuea_airbook_14_8gb.md', 'score': 2}

---
## Section 6: System Prompt (XML + Thai)

**ทำไม XML tags ภาษาอังกฤษ + คำสั่งภาษาไทย?**

- XML tags (`<rules>`, `<decision_tree>`) เป็นแค่โครงสร้าง — ทุกโมเดลเข้าใจ
- คำอธิบายเป็นภาษาไทย — เพราะ OpenThaiGPT ถูก train มาเข้าใจไทยดีที่สุด
- ได้ประโยชน์ทั้ง 2 ด้าน: โครงสร้างชัดเจน + ภาษาที่โมเดลถนัด

In [21]:
SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญประจำร้าน FahMai Electronics ทำหน้าที่ตอบคำถามลูกค้า

<rules>
- ใช้เฉพาะข้อมูลใน <context> ที่ให้มาเท่านั้น
- ห้ามใช้ความรู้ทั่วไปหรือข้อมูลภายนอก
- ห้ามแต่งคำตอบเอง (No Hallucination) เด็ดขาด
- ถ้าไม่แน่ใจ ให้ตอบ 9 แทนการเดา
</rules>

<decision_tree>
ขั้นที่ 1: ตรวจสอบว่าคำถามเกี่ยวกับร้าน FahMai หรือไม่
  → ถ้าคำถามไม่เกี่ยวกับร้าน FahMai, สินค้า, บริการ, นโยบาย,
    หรือแบรนด์ที่ร้านขาย (เช่น ถามเรื่องทั่วไป, อาหาร, การเมือง, กีฬา)
  → ตอบ ANSWER: 10 ทันที

ขั้นที่ 2: ค้นหาคำตอบใน <context>
  → อ่านข้อมูลอ้างอิงทั้งหมดอย่างละเอียด
  → หาตัวเลข, ชื่อรุ่น, สเปค, ราคา, นโยบาย ที่ตรงกับคำถาม

ขั้นที่ 3: ตัดสินใจ
  กรณี A — เจอคำตอบชัดเจนใน context:
    → เลือกตัวเลือก 1-8 ที่ตรงกับข้อมูลมากที่สุด

  กรณี B — คำถามเกี่ยวกับ FahMai แต่ไม่มีข้อมูลใน context:
    → ตอบ ANSWER: 9
    → ตัวอย่าง: ถามสเปคสินค้าที่ไม่มีในข้อมูล, ถามนโยบายที่ไม่ได้ระบุ

  กรณี C — คำถามไม่เกี่ยวกับ FahMai เลย:
    → ตอบ ANSWER: 10
    → ตัวอย่าง: ถามเรื่องร้านอาหาร, สภาพอากาศ, ข่าวทั่วไป
</decision_tree>

<output_format>
ตอบตามรูปแบบนี้เท่านั้น:

<reasoning>อธิบายสั้นๆ ว่าเจอข้อมูลอะไร หรือทำไมถึงเลือกคำตอบนี้</reasoning>
ANSWER: X

โดย X = หมายเลข 1-10
</output_format>

<examples>
ตัวอย่างที่ 1 — เจอคำตอบใน context:
<reasoning>ใน context ระบุว่า Watch S3 Ultra กันน้ำได้ 10 ATM ตรงกับตัวเลือก 3</reasoning>
ANSWER: 3

ตัวอย่างที่ 2 — เกี่ยวกับ FahMai แต่ไม่มีข้อมูล:
<reasoning>คำถามถามเรื่องสินค้ารุ่น X แต่ไม่มีข้อมูลรุ่นนี้ใน context</reasoning>
ANSWER: 9

ตัวอย่างที่ 3 — ไม่เกี่ยวกับ FahMai:
<reasoning>คำถามถามเรื่องร้านอาหาร ไม่เกี่ยวกับ FahMai Electronics</reasoning>
ANSWER: 10
</examples>"""

print("System Prompt loaded ✓")
print(f"Length: {len(SYSTEM_PROMPT)} chars")

System Prompt loaded ✓
Length: 1611 chars


In [22]:
def build_rag_prompt(question, choices, retrieved_chunks):
    """Build user prompt — XML structured"""
    context_parts = []
    for i, c in enumerate(retrieved_chunks):
        context_parts.append(
            f'<document source="{c["source"]}" index="{i+1}">\n{c["raw_text"]}\n</document>'
        )
    context = "\n\n".join(context_parts)

    choices_text = "\n".join(f"  {k}. {v}" for k, v in choices.items())

    return (
        f"<context>\n{context}\n</context>\n\n"
        f"<question>{question}</question>\n\n"
        f"<choices>\n{choices_text}\n</choices>\n\n"
        f"อ่าน <context> แล้วตอบตาม <decision_tree> และ <output_format>"
    )

---
## Section 7: Load Questions & Run

In [23]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS}")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in list(questions[0]['choices'].items())[:3]:
    print(f"  {k}. {v}")
print("  ...")

Loaded 100 questions, using first 100

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM
  ...


In [24]:
def run_improved_pipeline(questions, n=N_QUESTIONS):
    """Full pipeline: Extract → Expand → Hybrid → Route → VIP Bypass + Rerank+Siblings → LLM + Fallback"""
    predictions = {}
    details = {}  # เก็บ reasoning สำหรับ debug

    for i, q in enumerate(questions[:n]):
        print(f"\n{'='*50}")
        print(f"[Q{q['id']}] {q['question'][:80]}{'...' if len(q['question']) > 80 else ''}")

        # ==========================================
        # ROUND 1: Strict Search (หาแบบแม่นยำสูง)
        # ==========================================

        # Step 1: Extract + Expand + Hybrid Retrieve
        # ★ (จุดที่คุณเผลอลบทิ้งไป ผมใส่กลับมาให้แล้วครับ) ★
        candidate_idx = multi_query_hybrid_retrieve(q["question"], chunk_embeddings, k=TOP_K_FETCH)
        print(f"    Retrieved {len(candidate_idx)} candidates")

        # Step 1.5: Query Router — inject mandatory chunks ตาม intent
        candidate_idx = inject_mandatory_chunks(q["question"], candidate_idx, chunks)

        # Step 1.6: Product Focus — ถ้าถามสินค้าเฉพาะ ให้เน้นไฟล์นั้น
        candidate_idx = apply_product_focus(q["question"], candidate_idx, chunks)

        # ★ THE VIP BYPASS: แยกไฟล์ Policy ออกจาก Reranker ★
        policy_idx = [idx for idx in candidate_idx if not chunks[idx]["source"].startswith("products/")]
        product_idx = [idx for idx in candidate_idx if chunks[idx]["source"].startswith("products/")]

        # Step 2: Cross-Encoder Rerank (เฉพาะสินค้าเท่านั้น!)
        reranked_products = rerank_chunks(q["question"], product_idx, chunks, top_k=3)

        # จับนโยบายที่ Intent Router หามาให้ (เอาแค่ 4 ก้อนบนสุด) มารวมกับสินค้าที่ชนะ
        final_idx = list(dict.fromkeys(policy_idx[:4] + reranked_products))
        retrieved_1 = [chunks[idx] for idx in final_idx]

        print(f"    Final context [Round 1]: {len(retrieved_1)} chunks:")
        for c in retrieved_1:
            print(f"      → {c['source']}")

        # Step 3: LLM Generate (ใช้ openthaigpt เป็นกองหน้า)
        prompt_1 = build_rag_prompt(q["question"], q["choices"], retrieved_1)
        raw_1 = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_1},
        ], model="openthaigpt")  # ★ ระบุโมเดลหลักตรงนี้

        # Step 4: Parse with Pydantic
        result_1 = RAGAnswer.from_llm_response(raw_1)
        ans_1 = result_1.answer

        # หากตอบข้อ 1-8 (มีข้อมูลแน่ๆ) ให้จบการทำงานแล้วไปข้อต่อไป
        if ans_1 not in [9, 10]:
            predictions[q["id"]] = ans_1
            details[q["id"]] = result_1
            print(f"    Reasoning: {result_1.reasoning}")
            print(f"    => [Round 1] ANSWER: {ans_1}")
            time.sleep(0.3)
            continue

        # ==========================================
        # ROUND 2: Fallback (ถ้าตอบ 9, 10 ให้ปลดล็อกเงื่อนไขแล้วหาใหม่!)
        # ==========================================
        print(f"    [Alert] Model answered {ans_1}. Triggering Fallback Verification...")

        # 1. ปลดล็อก: ข้าม Product Focus ทิ้งไปเลย ให้ดึง Multi-Query ใหม่แบบกว้างๆ (k=20)
        wider_idx = multi_query_hybrid_retrieve(q["question"], chunk_embeddings, k=20)

        # ★ 2. VIP BYPASS สำหรับ Fallback (ใส่เพิ่มให้แล้วครับ) ★
        wider_policy = [idx for idx in wider_idx if not chunks[idx]["source"].startswith("products/")]
        wider_product = [idx for idx in wider_idx if chunks[idx]["source"].startswith("products/")]

        reranked_wider_prod = rerank_chunks(q["question"], wider_product, chunks, top_k=3)

        # รวมนโยบายเข้ากับสินค้าที่จัดอันดับใหม่
        wider_final_idx = list(dict.fromkeys(wider_policy[:4] + reranked_wider_prod))
        retrieved_2 = [chunks[idx] for idx in wider_final_idx]

        prompt_2 = build_rag_prompt(q["question"], q["choices"], retrieved_2)

        # 3. เปลี่ยนเป็น Pathumma 8B Think มาช่วยเป็น "ผู้ตรวจทาน (Auditor)"
        raw_2 = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_2},
        ], model="pathumma")

        result_2 = RAGAnswer.from_llm_response(raw_2)
        ans_2 = result_2.answer

        predictions[q["id"]] = ans_2
        details[q["id"]] = result_2

        if ans_2 != ans_1:
            print(f"    [Round 2 - SUCCESS] Changed answer from {ans_1} -> {ans_2}")
            print(f"    Reasoning: {result_2.reasoning}")
        else:
            print(f"    [Round 2 - CONFIRMED] Model insists on {ans_2}")
            print(f"    Reasoning: {result_2.reasoning}")

        time.sleep(0.3)

    return predictions, details

print("Pipeline ready ✓")

Pipeline ready ✓


In [25]:
# === RUN ===
improved_preds, run_details = run_improved_pipeline(questions)


[Q1] Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
    Expanded: 4 queries
      q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
      q2: Watch S3 Ultra กันน้ำ ค่า ATM
      q3: Watch S3 Ultra ความลึกการกันน้ำ ATM
      q4: Watch S3 Ultra ความสามารถในการกันน้ำ ATM
    Retrieved 15 candidates
    Intent: ['warranty'] → must include: ['return_policy', 'warranty_policy']
    Injected 4 mandatory chunks from: ['return_policy', 'warranty_policy']
    Product focus: ['products/WK-SW-001_wongkhojon_watch_s3_ultra.md']
      products/WK-SW-001_wongkhojon_watch_s3_ultra.md (score=6)
    Focus result: 7 primary + 3 others
    Reranked 3 → with siblings: 8 chunks
    Final context [Round 1]: 11 chunks:
      → policies/return_policy.md
      → policies/return_policy.md
      → policies/warranty_policy.md
      → products/WK-SW-001_wongkhojon_watch_s3_ultra.md
      → products/WK-SW-001_wongkhojon_watch_s3_ultra.md
      → products/WK-SW-001_wongkhojon_watch_s3_ultra.md
      → products/WK-SW-001_wongkhojon_wat

---
## Section 8: Evaluate vs Ground Truth

In [26]:
GROUND_TRUTH = {1: 5, 2: 7, 3: 2, 4: 6, 5: 6, 6: 8, 7: 1, 8: 4, 9: 4, 10: 7}

print(f"{'QID':>4}  {'Pred':>5}  {'GT':>4}  {'':>3}  Reasoning")
print("-" * 70)

correct = 0
for qid in sorted(GROUND_TRUTH.keys()):
    gt = GROUND_TRUTH[qid]
    pred = improved_preds.get(qid, "-")
    match = "✓" if pred == gt else "✗"
    if pred == gt:
        correct += 1
    reason = run_details[qid].reasoning[:50] if qid in run_details and run_details[qid].reasoning else "-"
    print(f"Q{qid:>3}  {pred:>5}  {gt:>4}  {match:>3}  {reason}")

print(f"\n{'='*70}")
print(f"Accuracy: {correct}/{len(GROUND_TRUTH)} = {correct/len(GROUND_TRUTH)*100:.1f}%")
print(f"Model: {LLM_MODEL}")

 QID   Pred    GT       Reasoning
----------------------------------------------------------------------
Q  1      5     5    ✓  ในเอกสารรายละเอียดสินค้า (index 5) และสเปคสินค้า (
Q  2      7     7    ✓  ในเอกสารสินค้า "SaiFah Pen Gen 2" ระบุราคาอย่างชัด
Q  3      2     2    ✓  ในเอกสารสเปคสินค้าของหูฟัง HeadPro X1 ระบุชัดเจนว่
Q  4      6     6    ✓  ในเอกสาร general_faq.md ระบุชัดเจนว่าขณะนี้ฟ้าใหม่
Q  5      6     6    ✓  ในเอกสาร "store_info/general_faq.md" ระบุว่าสามารถ
Q  6      8     8    ✓  ในเอกสารการชำระเงินของร้าน FahMai ระบุชัดเจนว่า "ฟ
Q  7      1     1    ✓  ในเอกสารที่อ้างอิงถึงสิ่งที่อยู่ในกล่องของ AirBook
Q  8      4     4    ✓  ในเอกสาร <document source="products/SF-SP-011_saif
Q  9      4     4    ✓  สายฟ้า Rugged R1 มีมาตรฐานกันน้ำระดับ IP69K ตามที่
Q 10      7     7    ✓  ในเอกสารสเปคของสายฟ้า โฟน X9 Pro ระบุว่ารองรับการใ

Accuracy: 10/10 = 100.0%
Model: openthaigpt


---
## Section 9: Debug — ดู reasoning ที่ตอบผิด

In [27]:
print("=== Questions ที่ตอบผิด ===")
for qid in sorted(GROUND_TRUTH.keys()):
    gt = GROUND_TRUTH[qid]
    pred = improved_preds.get(qid)
    if pred != gt and qid in run_details:
        d = run_details[qid]
        q = next(q for q in questions if q['id'] == qid)
        print(f"\n--- Q{qid}: {q['question']} ---")
        print(f"  Predicted: {pred} | Ground Truth: {gt}")
        print(f"  Reasoning: {d.reasoning}")
        print(f"  Raw: {d.raw_response[:200]}")

=== Questions ที่ตอบผิด ===


---
## Section 10: Write Submission

เปลี่ยน `N_QUESTIONS = 100` แล้ว re-run ทั้งหมดเพื่อ submit จริง

In [28]:
with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, improved_preds.get(qid, 1)])

print(f"submission.csv written ({len(questions)} rows)")
print(f"\nTip: เปลี่ยน N_QUESTIONS = 100 แล้ว Restart & Run All")

submission.csv written (100 rows)

Tip: เปลี่ยน N_QUESTIONS = 100 แล้ว Restart & Run All


---
## What's Next?

ถ้าอยากปรับเพิ่ม:
- **เปลี่ยน Reranker**: ลอง `BAAI/bge-reranker-base` แทน ms-marco
- **เปลี่ยน Embedding**: ลอง `intfloat/multilingual-e5-large`
- **Model Ensemble**: ใช้หลายโมเดล (typhoon + openthaigpt) แล้วโหวต
- **เพิ่มแบรนด์**: เปิด knowledge_base แล้วเพิ่มชื่อแบรนด์ใน `KNOWN_BRANDS`
- **HyDE**: ให้ LLM เดาคำตอบก่อน แล้วเอาไป embed ค้นหา